<a href="https://colab.research.google.com/github/guillermoLop/Data-Engineering/blob/main/GuillermoLopez_TP1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TP1 - Extracción


In [ ]:
!pip install pandas requests deltalake pyarrow

In [ ]:
import os
from datetime import datetime

import requests
import pandas as pd
from deltalake.writer import write_deltalake

In [ ]:
TMDB_TOKEN = "PEGAR_TOKEN_AQUI"

In [ ]:
BASE_URL = "https://api.themoviedb.org/3"

HEADERS = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}

In [ ]:
BASE_PATH = "data_lake/bronze/tmdb"

PATH_GENRES = f"{BASE_PATH}/movie_genres"
PATH_TRENDING = f"{BASE_PATH}/trending_movies_daily"

os.makedirs(PATH_GENRES, exist_ok=True)
os.makedirs(PATH_TRENDING, exist_ok=True)

print("Directorios creados correctamente.")

In [ ]:
def get_json(endpoint, params=None):
    """
    Consulta un endpoint de TMDb y devuelve la respuesta en formato JSON.
    """
    url = f"{BASE_URL}{endpoint}"

    response = requests.get(
        url,
        headers=HEADERS,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        raise Exception(
            f"Error al consultar API. Status: {response.status_code}. Respuesta: {response.text}"
        )

    return response.json()

In [ ]:
genres_json = get_json(
    "/genre/movie/list",
    params={"language": "es-AR"}
)

df_genres = pd.DataFrame(genres_json["genres"])

df_genres["fecha_extraccion"] = datetime.now().date().isoformat()
df_genres["fuente"] = "tmdb_api"
df_genres["endpoint"] = "/genre/movie/list"

df_genres.head()

In [ ]:
write_deltalake(
    PATH_GENRES,
    df_genres,
    mode="overwrite"
)

print("Extracción full guardada en Delta Lake.")

In [ ]:
now = datetime.now()

trending_json = get_json(
    "/trending/movie/day",
    params={"language": "es-AR"}
)

df_trending = pd.DataFrame(trending_json["results"])

df_trending["fecha_extraccion"] = now.date().isoformat()
df_trending["hora_extraccion"] = now.strftime("%H")
df_trending["fuente"] = "tmdb_api"
df_trending["endpoint"] = "/trending/movie/day"

df_trending.head()

In [ ]:
write_deltalake(
    PATH_TRENDING,
    df_trending,
    mode="append",
    partition_by=["fecha_extraccion", "hora_extraccion"]
)

print("Extracción incremental guardada en Delta Lake.")

In [ ]:
!find data_lake -maxdepth 5 -type d

In [ ]:
from deltalake import DeltaTable

dt_genres = DeltaTable(PATH_GENRES)
df_genres_check = dt_genres.to_pandas()

df_genres_check.head()

In [ ]:
dt_trending = DeltaTable(PATH_TRENDING)
df_trending_check = dt_trending.to_pandas()

df_trending_check.head()